# Load data and train/test split

Everyone uses this. Run these cells first.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay,
)
import matplotlib.pyplot as plt

df = pd.read_parquet("combined_preprocessed.parquet")
print(df.shape)


(6287183, 122)


In [ ]:
# Drop leakage columns (only known AFTER the flight — kept in preprocessed for EDA only)
leak_cols = [
    "DEP_DELAY",
    "ARR_DELAY",
    "DEP_DEL15",
    "WEATHER_DELAY",
    "NAS_DELAY",
    "SECURITY_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "DIVERTED",
    "DEST_AIRPORT_SEQ_ID",
]
X = df.drop(columns=[c for c in leak_cols + ["DELAY_CLASS"] if c in df.columns])
y = df["DELAY_CLASS"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Class distribution — train:")
print(y_train.value_counts(normalize=True).sort_index())


Helper function so everyone reports the same metrics.

In [ ]:
CLASS_NAMES = ['No Delay', '15-29 min', '30-59 min', '1-2 hr', '2+ hr']

def evaluate(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    print(f"--- {name} ---")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1:        {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted'):.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(cm, cmap='Blues')
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=10)
    ax.set_xticks(range(len(CLASS_NAMES)))
    ax.set_yticks(range(len(CLASS_NAMES)))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name} — Confusion Matrix')
    plt.tight_layout()
    plt.show()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='weighted', zero_division=0),
        "Recall": recall_score(y_test, y_pred, average='weighted', zero_division=0),
        "F1": f1_score(y_test, y_pred, average='weighted', zero_division=0),
        "AUC-ROC": roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted'),
    }

In [4]:
results = []

**most difficult task -- what features should we use??**

**what features we should drop?? one by one**

**and then, One-hot encoding so that we can add the destination in there**

---
# Wahid — Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# TODO (Wahid):
# 1. Create the model
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Naive Bayes', nb_model, X_test, y_test))


---
# Sam — Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# TODO (Sam):
# 1. Create model with: max_iter=1000, class_weight='balanced', random_state=42
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Logistic Regression', lr_model, X_test, y_test))

lr_model = Pipeline([
    ('scaler', StandardScaler(with_mean=False)), 
    ('clf', LogisticRegression(
        solver='saga',
        max_iter=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,  
    )),
])
lr_model.fit(X_train, y_train)

In [ ]:
results.append(evaluate('Logistic Regression', lr_model, X_test, y_test))

In [ ]:
# TODO (Sam): Feature importance plot
# 1. Get coefficients from lr_model.coef_[0]
# 2. Find the top 15 by absolute value
# 3. Plot as horizontal bar chart
coefs = lr_model.named_steps['clf'].coef_[0]
feature_names = X_train.columns

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'abs_coef': np.abs(coefs),
})

top15 = coef_df.nlargest(15, 'abs_coef').sort_values('coef')

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#d62728' if c > 0 else '#1f77b4' for c in top15['coef']]
ax.barh(top15['feature'], top15['coef'], color=colors)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Top 15 Features by |Coefficient| — Logistic Regression')

In [ ]:
## With Ridge Regression:
ridge_model = Pipeline([
    ('scaler', StandardScaler(with_mean=False)),
    ('clf', LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='saga',
        max_iter=200,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
    )),
])
ridge_model.fit(X_train, y_train)

In [ ]:
results.append(evaluate('Ridge Logistic Regression', ridge_model, X_test, y_test))

In [ ]:
from matplotlib.patches import Patch

clf = ridge_model.named_steps['clf']
coefs = clf.coef_[0]
feature_names = X_train.columns

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs,
    'abs_coef': np.abs(coefs),
})

top15 = coef_df.nlargest(15, 'abs_coef').sort_values('coef')

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#d62728' if c > 0 else '#1f77b4' for c in top15['coef']]
ax.barh(top15['feature'], top15['coef'], color=colors)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds, standardized)')
ax.set_title('Top 15 Features by |Coefficient| — Ridge Logistic Regression')

ax.legend(
    handles=[
        Patch(color='#d62728', label='Increases delay probability'),
        Patch(color='#1f77b4', label='Decreases delay probability'),
    ],
    loc='lower right',
)

plt.tight_layout()
plt.show()

---
# Aryan — Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO (Aryan):
# 1. Create model with: n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
# 2. Fit on X_train, y_train
# 3. results.append(evaluate('Random Forest', rf_model, X_test, y_test))


In [ ]:
# TODO (Aryan): Feature importance plot
# 1. Get importances from rf_model.feature_importances_
# 2. Find the top 15
# 3. Plot as horizontal bar chart


---
# Dennis — RNN

Needs `pip install tensorflow`.

In [ ]:
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout

# TODO (Dennis): Preprocessing
# 1. Scale X_train and X_test with StandardScaler
# 2. Reshape to 3D: (samples, 1, features) — RNN expects 3D input


In [ ]:
# TODO (Dennis): Build and train
# 1. Build Sequential: SimpleRNN(64) -> Dropout(0.3) -> Dense(32) -> Dense(5, softmax)
# 2. Compile with optimizer='adam', loss='sparse_categorical_crossentropy'
# 3. Fit with epochs=10, batch_size=1024, validation_split=0.1


In [ ]:
# TODO (Dennis): Evaluate
# 1. Get probabilities: y_prob = rnn_model.predict(X_test_rnn)  # shape (N, 5)
# 2. Get predictions: y_pred = y_prob.argmax(axis=1)
# 3. Print weighted metrics (accuracy, precision, recall, F1, AUC-ROC with multi_class='ovr')
# 4. Plot confusion matrix (5x5)
# 5. Append results dict to results list


---
# Compare all models

In [ ]:
comparison = pd.DataFrame(results).set_index('Model')
comparison.round(4)

In [ ]:
comparison.plot.bar(figsize=(10, 5), title='Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()